# Chap. 3 대규모 언어 모델 자세히 살펴보기

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

/Users/lemon7z/PycharmProjects/HandsOnLLM/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tokenizer = AutoTokenizer.from_pretrained('microsoft/Phi-3-mini-4k-instruct')

In [6]:
# model = AutoModelForCausalLM.from_pretrained('microsoft/Phi-3-mini-4k-instruct',device_map='mps',torch_dtype='auto', trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained("microsoft/Phi-3-mini-4k-instruct", torch_dtype=torch.float16, attn_implementation="eager").to("mps")

Loading weights: 100%|██████████| 195/195 [00:08<00:00, 22.28it/s]


In [10]:
generator = pipeline('text-generation', model=model, tokenizer=tokenizer, return_full_text=False,max_new_tokens=150, max_length=150,do_sample=False)

## Chap.3-1 트랜스포머 모델 개요

### 3.1.1.

In [11]:
prompt = 'Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened.'

In [12]:
output = generator(prompt)
print(output[0]['generated_text'])

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Mention that you've taken steps to prevent it in the future.


Email to Sarah:

Subject: Sincere Apologies for the Gardening Mishap


Dear Sarah,


I hope this message finds you well. I am writing to express my deepest apologies for the unfortunate incident that occurred in your garden yesterday.


As you know, I have always admired your green thumb and the lush oasis you've cultivated. It was with great regret that I accidentally damaged some of your prized plants while attempting to install a new irrigation system. The mishap happened when I misjudged the placement of the pip


### 3.1.2 정방향 계산의 구성 요소

In [13]:
print(model)

Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (qkv_proj): Linear(in_features=3072, out_features=9216, bias=False)
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear(in_features=3072, out_features=16384, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (activation_fn): SiLUActivation()
        )
        (input_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (resid_attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
      )
    )
    (norm): Phi3RMSNorm((3072,), eps=1e-05)
    (rotary_emb): Phi3RotaryEmbedding()
  )
  (lm_head): Linear(in_features=3072, out_featur

In [14]:
prompt = 'The capital of France is'

In [15]:
input_ids = tokenizer(prompt, return_tensors='pt').input_ids

In [16]:
input_ids = input_ids.to('mps')

In [17]:
model_output = model.model(input_ids)

In [18]:
lm_head_output = model.lm_head(model_output[0])

In [20]:
token_id = lm_head_output[0,-1].argmax(-1)
tokenizer.decode(token_id)

'Paris'

### 3.1.4 병렬 토큰 처리와 문맥의 크기

In [21]:
model_output[0].shape

torch.Size([1, 5, 3072])

In [22]:
lm_head_output.shape

torch.Size([1, 5, 32064])

In [23]:
prompt = 'Write a very long email apologizing to Sarah for the tragic gardening mishap. Explain how it happened.'

In [24]:
input_ids = tokenizer(prompt, return_tensors='pt').input_ids
input_ids = input_ids.to('mps')

In [25]:
%%timeit -n 1
generation_output = model.generate(input_ids=input_ids, max_new_tokens=100, use_cache=True)

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


8.47 s ± 1.48 s per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [26]:
%%timeit -n 1
generation_output = model.generate(input_ids=input_ids, max_new_tokens=100, use_cache=False)

15.6 s ± 5.17 s per loop (mean ± std. dev. of 7 runs, 1 loop each)


## Chap.3-2 트랜스포머 아키텍처의 최근 발전 사항